# MCP server demo
This notebook connects to the MCP server defined in this repository and
uses the result in a request to the OpenAI Chat Completions API.

In [ ]:
import os, json, urllib.request

SERVER_URL = 'http://localhost:3333/'

def mcp_request(method, params=None, id=1):
    payload = json.dumps({"jsonrpc": "2.0", "id": id, "method": method, "params": params or {}}).encode()
    req = urllib.request.Request(SERVER_URL, data=payload, headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(req) as resp:
        return json.load(resp)

mcp_request('list_resources')


In [ ]:
resource = mcp_request('read_resource', {'id': 'cat_fact'})
fact = resource['result']['data']['fact']
print(fact)


In [ ]:
api_key = os.environ.get('OPENAI_API_KEY')
if not api_key:
    raise RuntimeError('Set OPENAI_API_KEY environment variable')

openai_payload = json.dumps({
    'model': 'gpt-4o-mini',
    'messages': [
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user', 'content': f'Summarize this cat fact: {fact}'}
    ]
}).encode()

req = urllib.request.Request(
    'https://api.openai.com/v1/chat/completions',
    data=openai_payload,
    headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {api_key}'}
)
with urllib.request.urlopen(req) as resp:
    reply = json.load(resp)

reply['choices'][0]['message']['content']
